# EOPF GeoZarr Format — Interactive Exploration

This notebook demonstrates the usefulness of the **EOPF Zarr format** for cloud-native Sentinel-2 analysis:

- **No download required** — data streams directly from object storage
- **Analysis-ready** — the `eopf-zarr` xarray backend delivers properly georeferenced, scaled data
- **Resolution-aware** — request any resolution on open; bands are resampled on-the-fly
- **Interactive visualization** — explore imagery and indices with `hvplot`

We'll work over the **Ebro Delta, Spain** using Sentinel-2 L2A data.

## 1. Search the STAC catalog

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pystac_client

catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu/")

# Ebro Delta, Spain
bbox = [0.55, 40.55, 1.05, 40.90]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-04-01/2025-06-01",
    query={"eo:cloud_cover": {"lt": 20}},
)

items = list(search.items())
print(f"Found {len(items)} items")
for item in items:
    print(f"  {item.id}  cloud_cover={item.properties.get('eo:cloud_cover', 'N/A')}%")

Found 69 items
  S2C_MSIL2A_20250531T104041_N0511_R008_T31TCF_20250531T162214  cloud_cover=2.907349%
  S2C_MSIL2A_20250531T104041_N0511_R008_T31TCE_20250531T162214  cloud_cover=12.850912%
  S2C_MSIL2A_20250531T104041_N0511_R008_T31TBE_20250531T162214  cloud_cover=4.694793%
  S2C_MSIL2A_20250531T104041_N0511_R008_T30TYK_20250531T162214  cloud_cover=1.028097%
  S2B_MSIL2A_20250529T104619_N0511_R051_T31TCF_20250529T130912  cloud_cover=1.572946%
  S2B_MSIL2A_20250529T104619_N0511_R051_T31TCE_20250529T130912  cloud_cover=8.514763%
  S2B_MSIL2A_20250529T104619_N0511_R051_T31TBF_20250529T130912  cloud_cover=0.014701%
  S2B_MSIL2A_20250529T104619_N0511_R051_T31TBE_20250529T130912  cloud_cover=3.684168%
  S2B_MSIL2A_20250529T104619_N0511_R051_T30TYL_20250529T130912  cloud_cover=0.003709%
  S2B_MSIL2A_20250529T104619_N0511_R051_T30TYK_20250529T130912  cloud_cover=2.664812%
  S2A_MSIL2A_20250526T105051_N0511_R051_T31TCF_20250526T144258  cloud_cover=19.641463%
  S2A_MSIL2A_20250526T105051_N0511_R0

In [2]:
! pip install xarray-eopf

## 2. Open with the `eopf-zarr` xarray backend

The `xarray-eopf` package registers an `eopf-zarr` engine that:
- Reads the EOPF Zarr product structure
- Applies scale/offset automatically (returning physical reflectance values)
- Attaches proper CRS coordinates (`x`, `y` in projected metres + `spatial_ref`)
- Resamples all bands to the requested `resolution` on-the-fly

In [3]:
import xarray as xr

# Pick the least-cloudy scene
item = sorted(items, key=lambda i: i.properties.get('eo:cloud_cover', 999))[0]
url = item.assets['product'].href
print(f"Opening: {item.id}")
print(f"URL: {url}")

# Open at 60m resolution for fast exploration — change to 10m resolution for full detail
ds = xr.open_dataset(url, engine='eopf-zarr', resolution=60, chunks={})
ds

Opening: S2B_MSIL2A_20250506T103629_N0511_R008_T30TYK_20250506T125830
URL: https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/06/products/cpm_v256/S2B_MSIL2A_20250506T103629_N0511_R008_T30TYK_20250506T125830.zarr


<xarray.Dataset> Size: 332MB
Dimensions:      (y: 1830, x: 1830)
Coordinates:
    spatial_ref  int64 8B ...
  * x            (x) int64 15kB 699990 700050 700110 ... 809610 809670 809730
  * y            (y) int64 15kB 4499970 4499910 4499850 ... 4390290 4390230
    band         int64 8B ...
Data variables: (12/15)
    b08          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    cld          (y, x) uint8 3MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    snw          (y, x) uint8 3MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    b01          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    b02          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    b03          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    ...           ...
    b07          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    b09          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    b11          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    b12          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    b8a          (y, x) float64 27MB dask.array<chunksize=(305, 305), meta=np.ndarray>
    scl          (y, x) uint8 3MB dask.array<chunksize=(305, 305), meta=np.ndarray>
Attributes: (34)

Notice:
- **`x` / `y` coordinates** in UTM metres — no manual georeferencing needed
- **`spatial_ref`** variable carries the full CRS (EPSG:32632)
- **All bands** at a single consistent resolution via one `open_dataset` call
- Values are **float64 surface reflectance** (scale/offset already applied)

## 3. Clip to our area of interest

Reproject the bbox to UTM and slice — no download of the full 10980×10980 tile.

In [4]:
import pyproj

# Detect the actual CRS from the dataset's spatial_ref variable
spatial_ref = ds.get('spatial_ref', None)
if spatial_ref is not None and 'crs_wkt' in spatial_ref.attrs:
    detected_crs = pyproj.CRS.from_wkt(spatial_ref.attrs['crs_wkt'])
    epsg_code = detected_crs.to_epsg()
elif spatial_ref is not None and 'epsg_code' in spatial_ref.attrs:
    epsg_code = int(spatial_ref.attrs['epsg_code'])
else:
    epsg_code = 32631  # UTM zone 31N (Ebro Delta default)
print(f"Dataset CRS: EPSG:{epsg_code}")

transformer = pyproj.Transformer.from_crs("EPSG:4326", f"EPSG:{epsg_code}", always_xy=True)
x_min, y_min = transformer.transform(bbox[0], bbox[1])
x_max, y_max = transformer.transform(bbox[2], bbox[3])
print(f"AOI in EPSG:{epsg_code}: x=[{x_min:.0f}, {x_max:.0f}]  y=[{y_min:.0f}, {y_max:.0f}]")
print(f"Dataset x: [{float(ds.x.min()):.0f}, {float(ds.x.max()):.0f}]")
print(f"Dataset y: [{float(ds.y.min()):.0f}, {float(ds.y.max()):.0f}]")

# Handle both descending (north→south) and ascending y coordinates
if float(ds.y[0]) > float(ds.y[-1]):
    aoi = ds.sel(x=slice(x_min, x_max), y=slice(y_max, y_min))
else:
    aoi = ds.sel(x=slice(x_min, x_max), y=slice(y_min, y_max))
print(f"AOI dims: {dict(aoi.dims)}")
aoi

Dataset CRS: EPSG:32630
AOI in EPSG:32630: x=[800612, 841166]  y=[4494863, 4535558]
Dataset x: [699990, 809730]
Dataset y: [4390230, 4499970]
AOI dims: {'y': 86, 'x': 152}


<xarray.Dataset> Size: 1MB
Dimensions:      (y: 86, x: 152)
Coordinates:
    spatial_ref  int64 8B ...
  * x            (x) int64 1kB 800670 800730 800790 ... 809610 809670 809730
  * y            (y) int64 688B 4499970 4499910 4499850 ... 4494930 4494870
    band         int64 8B ...
Data variables: (12/15)
    b08          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    cld          (y, x) uint8 13kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    snw          (y, x) uint8 13kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    b01          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    b02          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    b03          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    ...           ...
    b07          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    b09          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    b11          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    b12          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    b8a          (y, x) float64 105kB dask.array<chunksize=(86, 152), meta=np.ndarray>
    scl          (y, x) uint8 13kB dask.array<chunksize=(86, 152), meta=np.ndarray>
Attributes: (34)

## 4. True-colour RGB composite

In [5]:
import hvplot.xarray  # noqa: registers hvplot accessor
import numpy as np

# Stack RGB bands and compute into memory (small AOI at 60 m)
rgb = xr.concat(
    [aoi['b04'], aoi['b03'], aoi['b02']],  # R, G, B
    dim='band'
).assign_coords(band=['red', 'green', 'blue']).compute()

lo = float(np.nanpercentile(rgb.values, 2))
hi = float(np.nanpercentile(rgb.values, 98))
rgb_stretched = ((rgb.clip(lo, hi) - lo) / (hi - lo)).clip(0, 1)

rgb_stretched.hvplot.rgb(
    x='x', y='y',
    bands='band',
    crs=f'EPSG:{epsg_code}',
    tiles='OSM',
    title=f"True Colour — {item.id[:25]}…",
    frame_width=500,
)

:Overlay
   .WMTS.I :WMTS   [Longitude,Latitude]
   .RGB.I  :RGB   [x,y]   (R,G,B)

## 5. NDVI — Normalised Difference Vegetation Index

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

Because the data is already in physical reflectance units, NDVI is a one-liner.

In [6]:
nir = aoi['b08'].compute()
red = aoi['b04'].compute()

ndvi = (nir - red) / (nir + red)
ndvi.name = 'NDVI'

ndvi.hvplot(
    x='x', y='y',
    cmap='RdYlGn',
    clim=(-0.2, 0.8),
    crs=f'EPSG:{epsg_code}',
    tiles='OSM',
    title='NDVI — vegetation health',
    frame_width=500,
    colorbar=True,
    alpha=0.7,
)

:Overlay
   .WMTS.I  :WMTS   [Longitude,Latitude]
   .Image.I :Image   [x,y]   (NDVI)

## 6. NDSI — Snow/Ice Index

$$\text{NDSI} = \frac{\text{Green} - \text{SWIR1}}{\text{Green} + \text{SWIR1}}$$

Useful for alpine scenes — values > 0.4 typically indicate snow/ice.

In [7]:
green = aoi['b03'].compute()
swir  = aoi['b11'].compute()

ndsi = (green - swir) / (green + swir)
ndsi.name = 'NDSI'

ndsi.hvplot(
    x='x', y='y',
    cmap='Blues_r',
    clim=(-0.5, 1.0),
    crs=f'EPSG:{epsg_code}',
    tiles='OSM',
    title='NDSI — snow / ice cover',
    frame_width=500,
    colorbar=True,
    alpha=0.7,
)

:Overlay
   .WMTS.I  :WMTS   [Longitude,Latitude]
   .Image.I :Image   [x,y]   (NDSI)

## 7. Side-by-side comparison with `+` layout

In [8]:
true_color = rgb_stretched.hvplot.rgb(
    x='x', y='y',
    bands='band',
    crs=f'EPSG:{epsg_code}',
    tiles='OSM',
    title=f"True Colour — {item.id[:25]}…",
    frame_width=380,
)
p_ndsi = ndsi.hvplot(
    x='x', y='y', cmap='Blues_r', clim=(-0.5, 1.0),
    crs=f'EPSG:{epsg_code}', tiles='OSM',
    title='NDSI', frame_width=380, colorbar=True, alpha=0.7,
)

true_color + p_ndsi

:Layout
   .Overlay.I  :Overlay
      .WMTS.I :WMTS   [Longitude,Latitude]
      .RGB.I  :RGB   [x,y]   (R,G,B)
   .Overlay.II :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [x,y]   (NDSI)

## 8. Comparing resolutions — zoom in at 10m resolution

One of the key benefits of the EOPF Zarr format is **lazy, resolution-on-demand** access.
Re-open at 10 m for a sharper look — only the AOI pixels are fetched.

In [9]:
ds_10m = xr.open_dataset(url, engine='eopf-zarr', resolution=10, chunks={})
if float(ds_10m.y[0]) > float(ds_10m.y[-1]):
    aoi_10m = ds_10m.sel(x=slice(x_min, x_max), y=slice(y_max, y_min))
else:
    aoi_10m = ds_10m.sel(x=slice(x_min, x_max), y=slice(y_min, y_max))

nir_10m = aoi_10m['b08'].compute()
red_10m = aoi_10m['b04'].compute()
ndvi_10m = (nir_10m - red_10m) / (nir_10m + red_10m)
ndvi_10m.name = 'NDVI'

p60 = ndvi.hvplot(
    x='x', y='y', cmap='RdYlGn', clim=(-0.2, 0.8),
    rasterize=True, crs=f'EPSG:{epsg_code}', tiles='OSM',
    title='NDVI @ 60m resolution', frame_width=380, colorbar=True, alpha=0.7,
)
p10 = ndvi_10m.hvplot(
    x='x', y='y', cmap='RdYlGn', clim=(-0.2, 0.8),
    rasterize=True, crs=f'EPSG:{epsg_code}', tiles='OSM',
    title='NDVI @ 10m resolution', frame_width=380, colorbar=True, alpha=0.7,
)

p60 + p10

:Layout
   .DynamicMap.I  :DynamicMap   []
      :Overlay
         .WMTS.I  :WMTS   [Longitude,Latitude]
         .Image.I :Image   [x,y]   (NDVI)
   .DynamicMap.II :DynamicMap   []
      :Overlay
         .WMTS.I  :WMTS   [Longitude,Latitude]
         .Image.I :Image   [x,y]   (NDVI)

## 9. Why EOPF Zarr?

| Feature | Traditional SAFE/GeoTIFF | EOPF Zarr |
|---|---|---|
| Format | Zip archive of JP2/TIFF | Cloud-native chunked Zarr |
| Download required? | Yes (1–8 GB) | No — stream only what you need |
| CRS/georef | Manual reprojection | Built-in via `spatial_ref` + coords |
| Scale/offset | Manual application | Auto-applied by backend |
| Multi-resolution | Separate files per resolution | Single open, `resolution=` param |
| xarray integration | Via rioxarray/stackstac | Native `eopf-zarr` engine |
| Parallel/Dask | Requires rechunking | Natively chunked |

Combined with a STAC catalog, this workflow scales from a laptop to a cloud cluster without changing a line of analysis code.